# Project 1 — AHP Method

Multi-Criteria Decision Analysis of European countries using the OECD Better Life Index.  
**Decision problem:** Selecting the best European country to live in from the perspective of a student based in Poznan, Poland.


In [1]:
import sys
import pathlib
import numpy as np
import pandas as pd

PROJECT_ROOT = pathlib.Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from common.config import DATASET_FILE, METADATA_FILE

df   = pd.read_csv(DATASET_FILE)
meta = pd.read_csv(METADATA_FILE)

print(f"Countries : {len(df)}")
print(f"Criteria  : {len(meta)}")
df.head()

Countries : 26
Criteria  : 8


,Country,Employment rate,Long-term unemployment rate,Personal earnings,Life expectancy,Life satisfaction,Employees working very long hours,Air pollution,Distance from Poznan (km)
0,Austria,72.0,1.3,53132.0,82.0,7.2,5.3,12.2,468.5
1,Belgium,65.0,2.3,54327.0,82.1,6.8,4.3,12.8,883.8
2,Czech Republic,74.0,0.6,29885.0,79.3,6.9,4.5,17.0,311.7
3,Denmark,74.0,0.9,58430.0,81.5,7.5,1.1,10.0,461.5
4,Estonia,74.0,1.2,30720.0,78.8,6.5,2.2,5.9,920.1


In [17]:
from ahp.hierarchy_setup import CATEGORIES, CATEGORY_CRITERIA, CRITERIA, DIRECTIONS

print("Criteria and their preference directions (+1 = gain, -1 = cost):\n")
for cat in CATEGORIES:
    print(f"Category: {cat}, criteria:")
    for crit in CATEGORY_CRITERIA[cat]:
        pref = "gain type" if DIRECTIONS[crit] == 1 else "cost type"
        print(f"{crit:<40s}  {pref}")
    print()

Criteria and their preference directions (+1 = gain, -1 = cost):

Category: Economic, criteria:
Employment rate                           gain type
Long-term unemployment rate               cost type
Personal earnings                         gain type

Category: Social/Health, criteria:
Life expectancy                           gain type
Life satisfaction                         gain type
Employees working very long hours         cost type

Category: Geography/Environment, criteria:
Air pollution                             cost type
Distance from Poznan (km)                 cost type



## AHP Hierarchy and Pairwise Comparison Matrices

The decision problem is structured as a 3-level hierarchy:

**Goal:** Find the best European country to live in for a student from Poznań.
```
GOAL
├── Economic
│       ├── Employment rate
│       ├── Long-term unemployment rate
│       └── Personal earnings
├── Social/Health
│       ├── Life expectancy
│       ├── Life satisfaction
│       └── Employees working very long hours
└── Geography/Environment
        ├── Air pollution
        └── Distance from Poznan (km)
```

At each level the DM provides pairwise comparisons using Saaty's 1-9 scale:

| Score | Meaning |
|-------|---------|
| 1 | Equal importance |
| 3 | Moderate importance |
| 5 | Strong importance |
| 7 | Very strong importance |
| 9 | Extreme importance |

**DM judgments (Goal level):**
- Economic is **strongly** more important than Social/Health (5×)
- Economic is **moderately** more important than Geography/Environment (3×)
- Social/Health is **slightly** more important than Geography/Environment (2×)

Note: the Goal matrix is intentionally inconsistent (CR=0.14 > 0.10) —
the three judgments are not perfectly transitive, which is natural in human judgment.

In [34]:
from ahp.dm_matrices import MATRICES
from ahp.weights import ahp_weights
from ahp.hierarchy_setup import CATEGORIES, CATEGORY_CRITERIA
import pandas as pd

LABELS = {
    "Goal":                  CATEGORIES,
    "Economic":              CATEGORY_CRITERIA["Economic"],
    "Social/Health":         CATEGORY_CRITERIA["Social/Health"],
    "Geography/Environment": CATEGORY_CRITERIA["Geography/Environment"],
}

results = {}

for name, A in MATRICES.items():
    labels = LABELS[name]
    r = ahp_weights(A)
    results[name] = r

    print(f"{name}")
    print("-" * 100)

    # pairwise matrix
    df_mat = pd.DataFrame(A, index=labels, columns=labels)
    print(df_mat.to_string(float_format=lambda x: f"{x:8.4f}"))
    print()

    # local weights with bar chart
    print("Local weights:")
    for lbl, w in zip(labels, r["weights"]):
        print(f"{lbl:<35s}  {w:.4f}")
    print()
    # consistency line
    status = "Satisfactory" if r["consistent"] else "Inconsistent (CR > 0.10)"
    print(f"λ_max={r['lambda_max']:.4f}  CI={r['CI']:.4f}  "
          f"RI={r['RI']:.4f}  CR={r['CR']:.4f}  {status}")
    print()

Goal
----------------------------------------------------------------------------------------------------
                       Economic  Social/Health  Geography/Environment
Economic                 1.0000         5.0000                 3.0000
Social/Health            0.2000         1.0000                 2.0000
Geography/Environment    0.3333         0.5000                 1.0000

Local weights:
Economic                             0.6571
Social/Health                        0.1963
Geography/Environment                0.1466

λ_max=3.1632  CI=0.0816  RI=0.5800  CR=0.1407  Inconsistent (CR > 0.10)

Economic
----------------------------------------------------------------------------------------------------
                             Employment rate  Long-term unemployment rate  Personal earnings
Employment rate                       1.0000                       0.5000             0.2000
Long-term unemployment rate           2.0000                       1.0000             0.2000
Per

### Global Criterion Weights

Local weights are multiplied by the parent category's Goal-level weight, 
then renormalised to sum to 1 across all 8 criteria.

A high global weight means performance on that criterion heavily drives 
the final country ranking.

In [44]:
from ahp.global_weights import compute_global_weights
from ahp.hierarchy_setup import CRITERION_CATEGORY

w_goal = results["Goal"]["weights"]
w_cats = {cat: results[cat]["weights"] for cat in CATEGORIES}
global_weights = compute_global_weights(w_goal, w_cats)

print(f"{'Criterion':<40s}  {'Category':<25s}  {'Weight':>7}")
print("-" * 80)
for crit, w in sorted(global_weights.items(), key=lambda x: -x[1]):
    print(f"  {crit:<38s}  {CRITERION_CATEGORY[crit]:<25s}  {w:.4f}")

Criterion                                 Category                    Weight
--------------------------------------------------------------------------------
  Personal earnings                       Economic                   0.4658
  Long-term unemployment rate             Economic                   0.1174
  Distance from Poznan (km)               Geography/Environment      0.1100
  Life satisfaction                       Social/Health              0.1059
  Employment rate                         Economic                   0.0739
  Employees working very long hours       Social/Health              0.0583
  Air pollution                           Geography/Environment      0.0367
  Life expectancy                         Social/Health              0.0321


### Inconsistency Analysis

The **Goal** matrix has CR > 0.10 and is therefore inconsistent.  
Below we reconstruct the perfectly consistent matrix implied by the eigenvector weights  
and identify where the DM's judgment deviates most.

In [48]:
from ahp.consistency import reconstruct_matrix, max_discrepancy

A_orig = MATRICES["Goal"]
A_rec  = reconstruct_matrix(results["Goal"]["weights"])
disc   = max_discrepancy(A_orig, A_rec, CATEGORIES)

print("Largest inconsistency in the Goal matrix:\n")
print(f"  Pair:                {disc['row_label']}  vs  {disc['col_label']}")
print(f"  DM judgment:         {disc['dm_value']:.4f}")
print(f"  Implied by weights:  {disc['rec_value']:.4f}")
print(f"  Difference:          {disc['diff']:.4f}")

Largest inconsistency in the Goal matrix:

  Pair:                Economic  vs  Social/Health
  DM judgment:         5.0000
  Implied by weights:  3.3472
  Difference:          1.6528


Interpretation: the DM rated Economic 5× more important than Social/Health, but the other two judgments imply only ~3.3×.



### Reconstructed vs Original Goal Matrix

The table below shows, for each pair, what the DM entered and what the eigenvector weights imply.  

In [54]:
import pandas as pd
from ahp.consistency import reconstruct_matrix
from ahp.dm_matrices import MATRICES

A_orig = MATRICES["Goal"]
A_rec  = reconstruct_matrix(results["Goal"]["weights"])

print("Original (DM):")
print(pd.DataFrame(A_orig, index=CATEGORIES, columns=CATEGORIES).to_string(float_format=lambda x: f"{x:.4f}"))

print("\nReconstructed (from weights):")
print(pd.DataFrame(A_rec,  index=CATEGORIES, columns=CATEGORIES).to_string(float_format=lambda x: f"{x:.4f}"))

Original (DM):
                       Economic  Social/Health  Geography/Environment
Economic                 1.0000         5.0000                 3.0000
Social/Health            0.2000         1.0000                 2.0000
Geography/Environment    0.3333         0.5000                 1.0000

Reconstructed (from weights):
                       Economic  Social/Health  Geography/Environment
Economic                 1.0000         3.3472                 4.4814
Social/Health            0.2988         1.0000                 1.3389
Geography/Environment    0.2231         0.7469                 1.0000


### Alternative Pairwise Comparisons — Scoring Method

To comply with Saaty's 1–9 scale, raw criterion values are **not** used directly.  
Instead, the absolute difference between two countries on each criterion is mapped  
to an integer score 1–9 via fixed thresholds (defined in `alternative_matrices.py`).  

Example for *Personal earnings*:

| Difference (USD) | Saaty score |
|-----------------|-------------|
| < 2 000          | 1 (equal)   |
| < 5 000          | 2           |
| < 10 000         | 3           |
| < 15 000         | 4           |
| < 20 000         | 5           |
| < 25 000         | 6           |
| < 30 000         | 7           |
| < 35 000         | 8           |
| ≥ 35 000         | 9           |

The direction (+1 gain / −1 cost) then determines which country gets the score and which gets its reciprocal.

### Alternative Scoring and Final Ranking

For each criterion, countries are compared pairwise using numerical 
differences mapped to Saaty scores via the thresholds defined in 
`alternative_matrices.py`. Each criterion produces a 26×26 comparison 
matrix; the principal eigenvector gives local alternative weights.

The country with the highest score is the model's top recommendation.

In [10]:
from ahp.scoring import compute_ahp_scores
import pandas as pd

scores = compute_ahp_scores(df, global_weights)

ranking = pd.DataFrame({
    "Country":   df["Country"].values,
    "AHP Score": scores,
}).sort_values("AHP Score", ascending=False).reset_index(drop=True)

ranking.index += 1
ranking.index.name = "Rank"
print(ranking)

              Country  AHP Score
Rank                            
1             Iceland   0.090508
2         Switzerland   0.088341
3          Luxembourg   0.077722
4         Netherlands   0.069335
5             Denmark   0.067769
6              Norway   0.055017
7             Germany   0.054431
8             Finland   0.046423
9             Austria   0.045818
10             Sweden   0.043187
11            Belgium   0.035973
12            Ireland   0.034122
13             Poland   0.031855
14     Czech Republic   0.030574
15     United Kingdom   0.030297
16            Estonia   0.024042
17             France   0.022690
18           Slovenia   0.021662
19          Lithuania   0.021550
20            Hungary   0.020718
21             Latvia   0.018473
22              Spain   0.017729
23              Italy   0.016349
24    Slovak Republic   0.015584
25           Portugal   0.012070
26             Greece   0.007764


### Conclusions

Under the DM's preferences — where **economic criteria dominate** 
(personal earnings alone carries ~47% of total weight) — **Iceland** 
and **Switzerland** emerge as the top two destinations, which offer
strong earnings, high employment, and good life satisfaction scores.

Countries close to Poznań like Czech Republic, Germany, Austria are ranked in 
the upper-middle tier: they benefit from the distance criterion but 
cannot compete on earnings with top countries.

Greece ranks last, penalised by low earnings and high long-term 
unemployment which are the two highest-weight criteria in the model.

> **Note:** the ranking is very sensitive to the Goal-level weights. Changing the DM's weights for Economics in favour of Geography/Environment would put in the lead nearby countries like Czech Republic or Germany.